## KNN Algorithm

#### Step 1: Visualizing the data as a starting point


As a starting point we used Class distripution to visulaize the data.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

#Our dataset is made up of many .txt files, so first we convert it into a dataset.
base_dir = "overall/overall"

def load_texts(folder, label):
    folder_path = os.path.join(base_dir, folder)
    texts=[]

    for filename in os.listdir(folder_path):
        if filename.endswith(".txt"):
            file_path= os.path.join(folder_path, filename)
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                texts.append(f.read())
    return pd.DataFrame({"text": texts, "label": label})

#We loaded each category of training data: fake, and legit seperately. 
fakeDF= load_texts("fake", 0)
legitDF = load_texts("real", 1) #Where 0 would indicate fake news, and 1 indicate real ones.

#Then combined them into one connected dataframe.
df = pd.concat([fakeDF, legitDF], ignore_index= True)

print(df.head())
print(df["label"].value_counts)

In [ ]:
counts = df['label'].value_counts().sort_index()

plt.figure(figsize=(6,4))
sns.barplot(x=counts.index, y=counts.values, palette="viridis")

plt.title("Category Distrobution: Fake vs. Legit News")
plt.show()

We can observe that the number of data entries for both fake and real news are fairly close in number. 

#### Step 2: Performing Term Frequency - Inverse Document Frequency (TF-IDF)

In an aim to find the words that are most important in diffrentiating between real and fake news.

In [ ]:
from sklearn.model_selection import train_test_split

#splitting the dataset to start, to work cleanly. 

X_train, X_test, y_train, y_test = train_test_split(
    df["text"], 
    df["label"], 
    test_size= 0.2, 
    random_state= 42, 
    stratify= df["label"]
)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

#Taking the shortcut to converting text values to numerical values. 
vectorizer = TfidfVectorizer(
    stop_words= "english", 
    max_features= 5000, #limiting the size
    ngram_range= (1,2) 
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.fit_transform(X_test)

In [ ]:
#Checking that everything is in order: 
print("Training vector shape:", X_train_vec.shape)
print("Testing vector shape:", X_test_vec.shape)

We made each news article -data entry- a numerical vector to fit it to KNN.

#### Step 3: Training K-Nearest Neighbours

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Train a k-NN classifier
knn = KNeighborsClassifier(n_neighbors=7) #Chose the value of 7 as it gave us the best possible accuracy.
knn.fit(X_train_vec, y_train)

# Predict and evaluate
y_pred = knn.predict(X_test_vec)
print(f"Test Accuracy (k=5): {accuracy_score(y_test, y_pred):.2f}")

#### Average‑Case Example

In [ ]:
example = ['''Several online posts claimed that new transportation plans were approved this week,
but no official announcement has been released. Some users said they heard about the changes from unnamed sources.''']

example_vec = vectorizer.transform(example)

fakeReal= "Real" if knn.predict(example_vec) else "Fake" 

print("KNN Predects that the Example is", fakeReal )

I chose this example because it is a neutral news text with slight uncertainty
making it suitable for testing the average case where algorithms may disagree

### Best Case Analysis (Time Complexity)
For the K-Nearest Neighbors (KNN) algorithm, the best-case scenario for time complexity is achieved by using partial selection (QuickSelect algorithm via `np.argpartition`). Instead of fully sorting all distances which takes $O(N \log N)$, it only partitions the array to find the top $K$ neighbors. This reduces the distance sorting time to $\Omega(N)$ in the best case.

In [ ]:
import numpy as np
import time

# Define dataset sizes for testing execution time
# Requirement: analyze performance over different n values [cite: 392, 400]
sizes = [100, 500, 1000, 2000, 5000]

def knn_best_case(x, X_train, y_train, k=5):
    """
    Implements KNN Best Case using Partial Selection (QuickSelect).
    Complexity: Omega(N) for sorting distances in the best case.
    """
    # Compute Euclidean distances between new sample and training data
    dists = np.linalg.norm(X_train - x, axis=1)
    
    # Partial selection using argpartition (Best Case scenario)
    # This avoids full sorting O(N log N) and achieves Omega(N)
    idx = np.argpartition(dists, k)[:k]
    
    # Majority vote to determine the predicted class
    return np.bincount(y_train[idx]).argmax()

print("KNN Best Case Execution Times:")
knn_best_times = []

for n in sizes:
    # Subset the training data based on size n 
    X_sub = np.array(X_train[:n]) 
    y_sub = np.array(y_train[:n])
    
    # Record the actual running time in milliseconds [cite: 392, 400]
    start_time = time.time()
    
    # Run the best case algorithm on a single test sample
    knn_best_case(np.array(X_test)[0], X_sub, y_sub)
    
    end_time = time.time()
    
    # Calculate execution time
    execution_time = end_time - start_time
    knn_best_times.append(execution_time)
    print(f"n={n}:\t {execution_time:.6f} seconds")